# Test all the trained models. User can define which model to use

Run the setup cell first, then run the prediction cell.

In [13]:
import certifi
import nltk
import ssl

ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

for resource in ("stopwords", "wordnet", "omw-1.4"):
    nltk.download(resource)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nileshsaumyadasa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/nileshsaumyadasa/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/nileshsaumyadasa/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
import joblib
import numpy as np
import re
from functools import lru_cache

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Choose "all" or one of: "svm", "logistic_regression", "naive_bayes"
MODEL_TO_TEST = "all"

MODEL_PATHS = {
    "svm": "/Users/nileshsaumyadasa/Documents/AIE410_Project/SVM_TFIDF/jasc_linear_svm.joblib",
    "logistic_regression": "/Users/nileshsaumyadasa/Documents/AIE410_Project/Logistic_Regression_TFIDF/jasc_logistic_regression.joblib",
    "naive_bayes": "/Users/nileshsaumyadasa/Documents/AIE410_Project/Multinomial_Naive_Bayes_TFIDF/jasc_multinomial_nb.joblib",
}

if MODEL_TO_TEST != "all" and MODEL_TO_TEST not in MODEL_PATHS:
    valid_models = ["all", *MODEL_PATHS]
    raise ValueError(f"Unknown model {MODEL_TO_TEST!r}. Choose from {valid_models}.")

# Choose how words are normalized: "lemma", "stem", or None
NORMALIZATION = "lemma"

# Load common English stop words, but keep negations because they are important
# for fault descriptions, such as "no leak", "not working", and "without damage"
STOP_WORDS = set(stopwords.words("english")) - {"no", "not", "nor", "without"}

# Select the word normalization method based on the NORMALIZATION setting
# Lemmatization converts words to their base forms; stemming removes word endings
# If None or an unrecognized option is provided, words remain unchanged
_norm = {
    "lemma": WordNetLemmatizer().lemmatize,
    "stem": PorterStemmer().stem
}.get(NORMALIZATION, lambda t: t)

# Cache previously normalized words to avoid processing repeated words again
_norm = lru_cache(maxsize=None)(_norm)

def preprocess_text(text):
    # Convert text to lowercase and replace punctuation and special characters
    # with spaces, then split the text into individual words
    tokens = re.sub(r"[\W_]+", " ", str(text).lower()).split()

    # Remove stop words, normalize the remaining words, and join them into
    # a single cleaned text string
    return " ".join(_norm(t) for t in tokens if t not in STOP_WORDS)


# Example maintenance text
example_text = [
    "Cabin door does not latch securely when closed."
]

# Apply the same preprocessing used during training
example_clean = [preprocess_text(t) for t in example_text]

def test_model(model_name, model_path):
    model = joblib.load(model_path)
    predicted = model.predict(example_clean)
    classifier = model.named_steps["classifier"]
    classes = np.asarray(classifier.classes_)

    if hasattr(model, "decision_function"):
        scores = np.asarray(model.decision_function(example_clean))
        score_type = "decision score"
    elif hasattr(model, "predict_proba"):
        scores = np.asarray(model.predict_proba(example_clean))
        score_type = "probability"
    else:
        raise AttributeError(
            f"{model_name} does not provide decision_function or predict_proba."
        )

    scores = np.atleast_2d(scores)[0]
    if scores.size != classes.size:
        raise ValueError(
            f"{model_name} returned {scores.size} scores for {classes.size} classes."
        )

    top5 = np.argsort(scores)[::-1][:5]
    print(f"\n=== {model_name} ===")
    print(f"Predicted JASC category: {predicted[0]}")
    print(f"Top 5 candidate categories ({score_type}):")
    for index in top5:
        print(f"  {classes[index]}  score={scores[index]:.3f}")


models_to_test = MODEL_PATHS if MODEL_TO_TEST == "all" else {
    MODEL_TO_TEST: MODEL_PATHS[MODEL_TO_TEST]
}

for model_name, model_path in models_to_test.items():
    test_model(model_name, model_path)


=== svm ===
Predicted JASC category: 52
Top 5 candidate categories (decision score):
  52  score=0.615
  82  score=-1.133
  25  score=-1.133
  32  score=-1.183
  78  score=-1.193

=== logistic_regression ===
Predicted JASC category: 52
Top 5 candidate categories (decision score):
  52  score=13.325
  25  score=8.349
  53  score=6.316
  33  score=6.007
  21  score=5.438

=== naive_bayes ===
Predicted JASC category: 52
Top 5 candidate categories (probability):
  52  score=0.836
  25  score=0.155
  32  score=0.004
  33  score=0.002
  53  score=0.002
